# 07 - Basic multi-objective optimization

This notebook introduces multi-objective optimization with explicit objective outputs, ACCIM parameters, and NSGA-II execution.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `OptimisationSimulation`
- `OptimisationSimulation.set_parameters`
- `OptimisationSimulation.set_problem`
- `OptimisationSimulation.run_optimisation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `optim.discover_available_outputs`
- `optim.select_outputs`
- `optim.apply_outputs_preflight`
- `optim.set_outputs_for_simulation`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Basic optimization: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import OptimisationSimulation

pd.set_option("display.max_columns", 160)


## Basic optimization: set project paths and discover IDF/EPW files

Here we declare the baseline project assets (IDFs, EPWs, and output folder).

For Basic optimization, this is the configuration anchor. If asset names change, update this block before touching execution cells.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_list`, `epw_list`, `RESULTS_DIRNAME`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_list = [
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
RESULTS_DIRNAME = "nb07_basic_optimization_results"

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## Basic optimization: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [ ]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


## Basic optimization: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [ ]:
building = ef.get_building(idf_list[0])


## Basic optimization: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
accim.utils.reduce_runtime(idf_object=building, timesteps=2)


## Basic optimization: initialize optimization simulation object

Create the optimization simulation object with objective context.

This creates the optimization simulation object that will manage evaluators, objectives, and algorithms. The next cells configure objectives and decision variables before solving.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `optim`.

In [ ]:
optim = OptimisationSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)


## Basic optimization: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:

df_meters_available, df_vars_available, outputs_meta = optim.discover_available_outputs(
    reduce_sim_time=True,
    prefer="testsimeplus",
)
print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


## Basic optimization: validate and apply output configuration

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `df_output_meter`: final meter table used by the simulation.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:

objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
df_meters_sel, _, _ = optim.select_outputs(
    meters=objective_meters,
    match="case_insensitive",
    on_missing="raise",
    suggest=True,
)

optim.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=None,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
)
optim.set_outputs_for_simulation(df_output_meter=df_meters_sel)


## Basic optimization: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.
- `minimize_outputs`: boolean vector indicating which objectives are minimized.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [ ]:
accis_parameters = {
	"CustAST_m": (0.15, 0.95),
	"CustAST_n": (8, 23),
	"CustAST_ASToffset": (1.0, 6.0),
	"CustAST_ASTall": (8, 16),
	"CustAST_ASTaul": (28, 38),
}

optim.set_parameters(accis_params_dict=accis_parameters)
optim.set_problem(minimize_outputs=[True, True])


## Basic optimization: estimate expected simulation workload

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

Key arguments used in this cell and why they matter:
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `epws`: list of EPW weather files used to run scenarios.

Variables created in this cell:
- `EVALUATIONS`: reused in subsequent workflow blocks.
- `POPULATION_SIZE`: reused in subsequent workflow blocks.
- `estimated`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
EVALUATIONS = 6
POPULATION_SIZE = 3

estimated = optim.estimate_optimisation_sims(
	evaluations=EVALUATIONS,
	population_size=POPULATION_SIZE,
	epws=epw_list,
)
print("Estimated simulations:", estimated)


## Basic optimization: run NSGA-II optimization and inspect Pareto flags

This cell runs the optimization algorithm with the current objectives and search settings.

Treat the returned dataframe as the optimization trace and confirm Pareto-related columns before decision-support steps.

Key arguments used in this cell and why they matter:
- `epws`: list of EPW weather files used to run scenarios.
- `out_dir`: output folder for generated artifacts and results.
- `evaluations`: total number of optimization evaluations.
- `population_size`: population size for the evolutionary algorithm.
- `algorithm`: optimization algorithm to execute.
- `processes`: number of parallel processes for simulation runs.
- `keep_df`: policy for storing output dataframes.
- `keep_sim_files`: policy for preserving simulation files.
- `keep_sim_files_batch_size`: batch size used when preserving simulation files.
- `dropna`: defines this setting for the current call.

Variables created in this cell:
- `outputs_optim`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check objective columns and the `pareto-optimal` indicator before post-analysis.
- Confirm the selected algorithm and evaluation budget were applied as intended.

In [ ]:

outputs_optim = optim.run_optimisation(
    epws=epw_list,
    out_dir=RESULTS_DIRNAME,
    evaluations=EVALUATIONS,
    population_size=POPULATION_SIZE,
    algorithm="NSGAII",
    processes=1,
    keep_df="all",
    keep_sim_files="non-dominated",
    keep_sim_files_batch_size=POPULATION_SIZE,
)
print("Rows:", len(outputs_optim))
if "pareto-optimal" in outputs_optim.columns:
    print(outputs_optim["pareto-optimal"].value_counts(dropna=False))
print(outputs_optim.head())


### Reading the result

Inspect objective columns and `pareto-optimal` status before moving to compromise or robustness analysis.

## Basic optimization: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [ ]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


## Basic optimization: preview cleanup actions with dry-run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
# Force-remove every generated artifact inside the dedicated result folder,
# including generated IDF files.
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=False,
	remove_empty_dirs=True,
)
